-----------------------
# Loan Credit Analysis
-----------------------

In [2]:
#Importing libaries needed for the project
import pandas as pd
import numpy as np

In [3]:
#Importing the dataset into the IDE
loan_df = pd.read_csv("C:\\Users\\User\\OneDrive\\Loan Credit Analysis\\loan_risk_prediction_dataset.csv")


In [4]:
loan_df.head(10)

,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved
0,56,48353.0,31258.0,675.0,20,Female,High School,Houston,Unemployed,0
1,69,57462.0,23262.0,586.0,6,Male,High School,San Francisco,Self-Employed,0
2,46,44219.0,26530.0,781.0,26,Male,PhD,Houston,Self-Employed,1
3,32,56307.0,11531.0,549.0,11,Male,NaN,New York,Unemployed,0
4,60,37034.0,27871.0,500.0,19,Female,High School,Chicago,Unemployed,0
5,25,47886.0,18106.0,835.0,13,Male,Masters,New York,Salaried,1
6,38,54748.0,25374.0,760.0,9,Female,High School,New York,Self-Employed,1
7,56,NaN,6279.0,599.0,22,Male,PhD,New York,Unemployed,0
8,36,25918.0,25041.0,777.0,29,Female,Bachelors,San Francisco,Unemployed,0
9,40,43415.0,2065.0,382.0,30,Female,High School,San Francisco,Self-Employed,0


# Checking null Values

In [9]:
#lets check the null values in the dataset
loan_df.isnull().sum()

Age                  0
Income             196
LoanAmount           0
CreditScore        194
YearsExperience      0
Gender               0
Education          198
City                 0
EmploymentType       0
LoanApproved         0
dtype: int64

In [11]:
#loan and credit score are numerical values, the resolve would be that impute using mean function.
loan_df[["Income","CreditScore"]] = loan_df[["Income","CreditScore"]].fillna(loan_df[["Income","CreditScore"]].mean())
loan_df.isnull().sum()

Age                  0
Income               0
LoanAmount           0
CreditScore          0
YearsExperience      0
Gender               0
Education          198
City                 0
EmploymentType       0
LoanApproved         0
dtype: int64

In [13]:
loan_df["Education"] = loan_df["Education"].fillna(loan_df["Education"].mode()[0])
loan_df.isnull().sum()

Age                0
Income             0
LoanAmount         0
CreditScore        0
YearsExperience    0
Gender             0
Education          0
City               0
EmploymentType     0
LoanApproved       0
dtype: int64

# Assessing Datatypes of our dataset

In [20]:
#Checking datatypes that will help us understand what elements are are in my features for analysis.
loan_df.dtypes

Age                 int64
Income              int32
LoanAmount          int32
CreditScore         int32
YearsExperience     int64
Gender             object
Education          object
City               object
EmploymentType     object
LoanApproved        int64
dtype: object

In [34]:
#Through experience, float datatypes gives error when working with Power BI 
#so i must have the values be integers, instead of float
loan_df[["Income","LoanAmount","CreditScore"]] = loan_df[["Income","LoanAmount","CreditScore"]].round().astype(int)
loan_df.dtypes

Age                 int64
Income              int32
LoanAmount          int32
CreditScore         int32
YearsExperience     int64
Gender             object
Education          object
City               object
EmploymentType     object
LoanApproved        int64
dtype: object

# Checking for duplicates

In [37]:
#Checking for the duplicated values in the dataset.
print(f'There are {loan_df.duplicated().sum()} number of duplicates in the dataset.')


There are 0 number of duplicates in the dataset.


# Feature engineering


In [40]:
#gender mapping for the machine learning process in the later stages
loan_df["GenderMapping"] = loan_df["Gender"].map({
    'Male': 1,
    "Female": 0
})

#creating a dept to income ratio
loan_df["DeptToIncome(%)"] = (
    loan_df["LoanAmount"] / loan_df["Income"] * 100
)

#grouping the ages of the lenders
loan_df["Age_group"] = pd.cut(
    loan_df["Age"], 
    bins=[18,30,45,60,100],
    labels = [
        'young',
        'adulting',
        'middle_age',
        'senior'
    ]
)

In [42]:
loan_df.head()

,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved,GenderMapping,DeptToIncome(%),Age_group
0,56,48353,31258,675,20,Female,High School,Houston,Unemployed,0,0,64.645420,middle_age
1,69,57462,23262,586,6,Male,High School,San Francisco,Self-Employed,0,1,40.482406,senior
2,46,44219,26530,781,26,Male,PhD,Houston,Self-Employed,1,1,59.996834,middle_age
3,32,56307,11531,549,11,Male,Bachelors,New York,Unemployed,0,1,20.478804,adulting
4,60,37034,27871,500,19,Female,High School,Chicago,Unemployed,0,0,75.257871,middle_age


In [64]:
#Checking for outliers
#first I will look at the age column to see if there are no outlier variances that may affect the accuracy of our data
columns = loan_df[["Age", "Income", "LoanAmount", "CreditScore"]]
#for loop that will look at each column for outliers
for col in columns:
    Q1 = loan_df[col].quantile(0.25)
    Q3 = loan_df[col].quantile(0.75)
    #getting the IQR 
    IQR = Q3 - Q1
    #Getting the lower and upper bound limits
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    #Getting the outliers
    outliers = loan_df[(loan_df[col] < lower) | (loan_df[col] > upper)]

    #print the outcome
    print(f'\n{col}')
    print(f'Q1: {Q1}')
    print(f'Q3: {Q3}')
    print(f'Lower bound: {lower}')
    print(f'Upper bound: {upper}')
    print(f'Outliers: {len(outliers)}')
    




Age
Q1: 31.0
Q3: 56.0
Lower bound: -6.5
Upper bound: 93.5
Outliers: 0

Income
Q1: 39990.25
Q3: 59447.0
Lower bound: 10805.125
Upper bound: 88632.125
Outliers: 47

LoanAmount
Q1: 14455.25
Q3: 25326.75
Lower bound: -1852.0
Upper bound: 41634.0
Outliers: 32

CreditScore
Q1: 440.0
Q3: 706.0
Lower bound: 41.0
Upper bound: 1105.0
Outliers: 0


In [84]:
#We have two features with outliers
#investigate the income column
income = loan_df["Income"]
income_Q1 = income.quantile(0.25)
income_Q3 = income.quantile(0.75)

income_IQR = income_Q3 - income_Q1

income_low = income_Q1 - 1.5 * income_IQR
income_upper = income_Q3 + 1.5 * income_IQR

income_outliers = loan_df[(loan_df["Income"] < income_low) | (loan_df["Income"] > income_upper)]

print(income_outliers.shape)
income_outliers.head(25)

#this data tells me whats the low range of income and the high range of income.

(47, 13)


,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved,GenderMapping,DeptToIncome(%),Age_group
138,18,9728,31123,817,25,Male,Bachelors,San Francisco,Salaried,0,1,319.932155,NaN
313,28,95929,15957,461,22,Male,High School,Houston,Unemployed,0,1,16.634177,young
390,69,129,23617,325,33,Female,PhD,San Francisco,Self-Employed,0,0,18307.751938,senior
403,41,90270,12114,551,3,Male,High School,Houston,Self-Employed,0,1,13.419741,adulting
659,33,10035,6256,517,26,Female,Bachelors,Houston,Self-Employed,0,0,62.341804,adulting
853,49,91490,25986,362,15,Male,High School,Houston,Salaried,0,1,28.403104,middle_age
984,41,89503,23546,479,5,Male,Bachelors,Chicago,Salaried,0,1,26.307498,adulting
1058,20,10387,18847,636,9,Female,Bachelors,Chicago,Self-Employed,0,0,181.447964,young
1069,45,89676,29692,693,6,Male,PhD,Chicago,Self-Employed,1,1,33.110308,adulting
1100,55,10594,29790,773,5,Female,Masters,Chicago,Salaried,0,0,281.196904,middle_age


In [86]:
#We have two features with outliers
#investigate the income column
amount = loan_df["LoanAmount"]
amount_Q1 = amount.quantile(0.25)
amount_Q3 = amount.quantile(0.75)

amount_IQR = amount_Q3 - amount_Q1

amount_low = amount_Q1 - 1.5 * amount_IQR
amount_upper = amount_Q3 + 1.5 * amount_IQR

amount_outliers = loan_df[(loan_df["LoanAmount"] < amount_low) | (loan_df["LoanAmount"] > amount_upper)]

print(amount_outliers.shape)
amount_outliers.head(25)

(32, 13)


,Age,Income,LoanAmount,CreditScore,YearsExperience,Gender,Education,City,EmploymentType,LoanApproved,GenderMapping,DeptToIncome(%),Age_group
17,39,35401,-4250,615,32,Female,PhD,Chicago,Unemployed,0,0,-12.005311,adulting
93,41,73864,-5728,525,22,Male,High School,Chicago,Unemployed,0,1,-7.754793,adulting
439,68,32242,-2711,575,24,Male,High School,Chicago,Unemployed,0,1,-8.408287,senior
451,39,31791,41709,664,12,Male,PhD,New York,Unemployed,0,1,131.197509,adulting
453,69,28985,43891,410,19,Male,Bachelors,Houston,Unemployed,1,1,151.426600,senior
609,52,39402,-2533,385,21,Female,High School,New York,Self-Employed,0,0,-6.428608,middle_age
743,28,45116,-2361,575,29,Male,PhD,Chicago,Self-Employed,0,1,-5.233177,young
818,30,52945,-4933,849,22,Male,PhD,San Francisco,Self-Employed,1,1,-9.317216,young
833,20,53789,47185,619,29,Male,Masters,Chicago,Self-Employed,1,1,87.722397,young
1119,37,86623,-2314,476,29,Male,PhD,Chicago,Self-Employed,0,1,-2.671346,adulting


In [82]:
loan_df["IncomeOutliers"] = np.where(income_outliers, "yes" , "no")

0        True
1       False
2       False
3       False
4       False
        ...  
4995    False
4996    False
4997    False
4998    False
4999    False
Name: LoanAmount, Length: 5000, dtype: bool


In [96]:
#including the income outlier feature in the dataframe
condition = (
    (loan_df["Income"] < income_low) | (loan_df["Income"] > income_upper)
)
loan_df["IncomeOutlier"] = np.where(condition, "yes", "no")
print(loan_df)

      Age  Income  LoanAmount  CreditScore  YearsExperience  Gender  \
0      56   48353       31258          675               20  Female   
1      69   57462       23262          586                6    Male   
2      46   44219       26530          781               26    Male   
3      32   56307       11531          549               11    Male   
4      60   37034       27871          500               19  Female   
...   ...     ...         ...          ...              ...     ...   
4995   24   36780       23383          575               23    Male   
4996   66   99146        9760          306               14    Male   
4997   26   58100       18230          311               10  Female   
4998   53   58513       12373          813               23    Male   
4999   36   58928       23615          648               10    Male   

        Education           City EmploymentType  LoanApproved  GenderMapping  \
0     High School        Houston     Unemployed             0      

In [100]:
condition2 = (
    (loan_df["LoanAmount"] < amount_low) | (loan_df["LoanAmount"] > amount_upper)
)
loan_df["loanOutlier"] = np.where(condition2, "yes", "no")
print(loan_df)

      Age  Income  LoanAmount  CreditScore  YearsExperience  Gender  \
0      56   48353       31258          675               20  Female   
1      69   57462       23262          586                6    Male   
2      46   44219       26530          781               26    Male   
3      32   56307       11531          549               11    Male   
4      60   37034       27871          500               19  Female   
...   ...     ...         ...          ...              ...     ...   
4995   24   36780       23383          575               23    Male   
4996   66   99146        9760          306               14    Male   
4997   26   58100       18230          311               10  Female   
4998   53   58513       12373          813               23    Male   
4999   36   58928       23615          648               10    Male   

        Education           City EmploymentType  LoanApproved  GenderMapping  \
0     High School        Houston     Unemployed             0      